# Datakvalitet – strømforbruk

Kontrollerer **hele `data/consumption_processed.csv`**, uten å endre datafilen.
En observasjon forventes å være én time per **kommunenummer og forbruksgruppe**.
Forbruket er aggregert, og antall målepunkter er et antall per observasjon, ikke måler-ID-er.
Vi kan derfor ikke telle unike fysiske målere eller summere antallet over timer.

Filen leses i bolker. Kompakte tidsserier beholdes i minnet for å kontrollere hull,
duplikate nøkler og endringer over bolkgrenser. Kjør cellene i rekkefølge;
full gjennomgang av omtrent 4 GB kan ta flere minutter og bruke noen GB RAM.
Rapportene skiller sikre strukturfeil fra statistiske varsler som krever vurdering.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
SOURCE = ROOT / "data" / "consumption_processed.csv"
if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)
CHUNK_SIZE = 200_000
SERIES = ["KOMMUNENUMMER", "FORBRUKSGRUPPE"]
POINTS = "ANTALL_MÅLEPUNKT"
HOUR_NS = pd.Timedelta(hours=1).value
# Varselgrenser, ikke dokumenterte fysiske maksimumsverdier.
PRIVATE_KWH_PER_HOUR = 25
POINT_CHANGE_FRACTION = 0.20
POINT_CHANGE_MINIMUM = 10
print(f"Kilde: {SOURCE} ({SOURCE.stat().st_size / 1e9:.2f} GB)")


Kilde: C:\Users\jsand\OneDrive - OsloMet\NETTSKY\SKOLE\26-27\SEMESTER 1\ACIT4510\energy\data\consumption_processed.csv (4.12 GB)


## Innlesing, manglende verdier og gyldighet

Semikolon og desimalkomma tolkes eksplisitt. Kommunenummer beholdes som tekst
med fire siffer. Tom tekst og blanktegn regnes som manglende; øvrige ugyldige tall
og datoer telles separat. Tidsstempler må ha eksplisitt UTC-offset.
Tidene konverteres til UTC for sammenligning og til `Europe/Oslo` for kalenderkontroll.


In [2]:
missing = Counter()
issues = Counter()
examples = {}
parts = defaultdict(list)
geography_parts = []
hash_parts = []
total_rows = 0

for chunk_number, raw in enumerate(pd.read_csv(
    SOURCE, sep=";", encoding="utf-8-sig", dtype="string",
    keep_default_na=False, chunksize=CHUNK_SIZE
), start=1):
    raw = raw.apply(lambda column: column.str.strip()).replace("", pd.NA)
    required = SERIES + ["KOMMUNE", "STARTTID", "SLUTTID", "VOLUM_KWH", POINTS]
    if not set(required).issubset(raw.columns):
        raise ValueError(f"Mangler kolonner: {set(required) - set(raw.columns)}")
    missing.update(raw.isna().sum().to_dict())
    # Hashene brukes bare til å finne kandidater. Like rader bekreftes senere.
    hash_parts.append(pd.util.hash_pandas_object(raw, index=False).to_numpy())
    df = raw.copy()
    df["KOMMUNENUMMER"] = raw["KOMMUNENUMMER"].str.zfill(4)
    masks = {"ugyldig_kommunenummer": ~df["KOMMUNENUMMER"].str.fullmatch(r"\d{4}", na=False)}
    for col in ["STARTTID", "SLUTTID"]:
        has_offset = raw[col].str.contains(r"(?:[+-]\d{2}:?\d{2}|Z)$", na=False)
        df[col] = pd.to_datetime(raw[col].where(has_offset), utc=True, errors="coerce", format="mixed").dt.as_unit("ns")
        masks[f"ugyldig_{col}"] = raw[col].notna() & df[col].isna()
        oslo = df[col].dt.tz_convert("Europe/Oslo")
        original_offset = raw[col].str.extract(r"([+-]\d{2}:?\d{2}|Z)$", expand=False).str.replace(":", "", regex=False).replace("Z", "+0000")
        masks[f"feil_oslo_offset_{col}"] = df[col].notna() & original_offset.ne(oslo.dt.strftime("%z"))
    for col in ["VOLUM_KWH", POINTS]:
        df[col] = pd.to_numeric(raw[col].str.replace(",", ".", regex=False), errors="coerce").astype("float64")
        masks[f"ugyldig_{col}"] = raw[col].notna() & ~np.isfinite(df[col])
    v, n = df["VOLUM_KWH"], df[POINTS]
    duration = (df["SLUTTID"] - df["STARTTID"]).dt.total_seconds() / 3600
    masks.update({
        "varighet_ikke_en_time": duration.notna() & duration.ne(1),
        "start_utenfor_hel_time": df["STARTTID"].notna() & df["STARTTID"].ne(df["STARTTID"].dt.floor("h")),
        "negativt_forbruk": v.lt(0),
        "negative_målepunkter": n.lt(0),
        "målepunkter_ikke_heltall": np.isfinite(n) & n.mod(1).ne(0),
        "null_målepunkter": n.eq(0),
        "forbruk_uten_målepunkter": n.eq(0) & v.ne(0) & np.isfinite(v),
        "nullforbruk_med_målepunkter": v.eq(0) & n.gt(0),
        "høyt_privat_forbruk_per_punkt": df["FORBRUKSGRUPPE"].eq("Privat") & n.gt(0) & (v / n / duration.where(duration.gt(0))).gt(PRIVATE_KWH_PER_HOUR),
    })
    local = df["STARTTID"].dt.tz_convert("Europe/Oslo")
    for col, expected in {"YEAR": local.dt.year, "MONTH": local.dt.month,
                          "DAY": local.dt.day, "HOUR": local.dt.hour,
                          "DAY_OF_WEEK": local.dt.dayofweek}.items():
        if col in raw:
            masks[f"feil_{col}"] = pd.to_numeric(raw[col], errors="coerce").ne(expected) & df["STARTTID"].notna()
    for label, mask in masks.items():
        mask = mask.fillna(False)
        issues[label] += int(mask.sum())
        if mask.any() and label not in examples:
            examples[label] = raw.loc[mask].head(5)
    valid_key = df[SERIES].notna().all(axis=1) & df["STARTTID"].notna() & ~masks["ugyldig_kommunenummer"]
    issues["rader_uten_gyldig_tidsserienøkkel"] += int((~valid_key).sum())
    valid = df.loc[valid_key].copy()
    valid["t"] = valid["STARTTID"].astype("int64")
    for key, group in valid.groupby(SERIES, sort=False):
        # Separate arrays preserve nanosecond timestamps exactly.
        parts[key].append((group["t"].to_numpy(), group["VOLUM_KWH"].to_numpy(), group[POINTS].to_numpy()))
    geography_parts.append(valid.groupby(["KOMMUNENUMMER", "KOMMUNE"], dropna=False)["STARTTID"].agg(["min", "max", "size"]).reset_index())
    total_rows += len(raw)
    if chunk_number % 10 == 0:
        print(f"Lest {total_rows:,} rader", flush=True)

print(f"Ferdig: {total_rows:,} rader, {len(parts):,} tidsserier")
display(pd.Series(missing, name="manglende").to_frame())
display(pd.Series(issues, name="antall_rader").to_frame())
# Eksempel: display(examples["negativt_forbruk"])


Lest 2,000,000 rader
Lest 4,000,000 rader
Lest 6,000,000 rader
Lest 8,000,000 rader
Lest 10,000,000 rader
Lest 12,000,000 rader
Lest 14,000,000 rader
Lest 16,000,000 rader
Lest 18,000,000 rader
Lest 20,000,000 rader
Lest 22,000,000 rader
Lest 24,000,000 rader
Lest 26,000,000 rader
Lest 28,000,000 rader
Lest 30,000,000 rader
Lest 32,000,000 rader
Lest 34,000,000 rader
Lest 36,000,000 rader
Lest 38,000,000 rader
Lest 39,951,726 rader
Ferdig: 39,951,726 rader, 1,074 tidsserier


,manglende
STARTTID,0
SLUTTID,0
KOMMUNE,0
KOMMUNENUMMER,0
FORBRUKSGRUPPE,0
VOLUM_KWH,0
ANTALL_MÅLEPUNKT,0
YEAR,0
MONTH,0
DAY,0


,antall_rader
ugyldig_kommunenummer,0
ugyldig_STARTTID,0
feil_oslo_offset_STARTTID,0
ugyldig_SLUTTID,0
feil_oslo_offset_SLUTTID,0
ugyldig_VOLUM_KWH,0
ugyldig_ANTALL_MÅLEPUNKT,0
varighet_ikke_en_time,0
start_utenfor_hel_time,0
negativt_forbruk,3347


## Duplikater

Helt like rader kontrolleres på tvers av hele filen, inkludert bolkgrenser.
Hashkollisjoner håndteres ved å lese kandidatene på nytt og sammenligne alle felter.
Sammenligningen gjelder innhold etter trimming av blanktegn. Flere rader med samme
kommunenummer, gruppe og UTC-starttid rapporteres også, selv om forbruket er forskjellig.


In [3]:
hash_counts = pd.Series(np.concatenate(hash_parts)).value_counts()
candidate_hashes = set(hash_counts.index[hash_counts.gt(1)])
del hash_parts, hash_counts
duplicate_rows = pd.DataFrame()
exact_duplicate_excess = 0
if candidate_hashes:
    candidates = []
    for raw in pd.read_csv(SOURCE, sep=";", encoding="utf-8-sig", dtype="string", keep_default_na=False, chunksize=CHUNK_SIZE):
        raw = raw.apply(lambda column: column.str.strip()).replace("", pd.NA)
        h = pd.util.hash_pandas_object(raw, index=False)
        candidates.append(raw.loc[h.isin(candidate_hashes)])
    candidates = pd.concat(candidates, ignore_index=True)
    exact_duplicate_excess = int(candidates.duplicated().sum())
    duplicate_rows = candidates.loc[candidates.duplicated(keep=False)]
    del candidates
print("Helt like overskytende rader:", exact_duplicate_excess)
display(duplicate_rows.head(20))


Helt like overskytende rader: 0


""


## Manglende timer, målepunkter og statistiske forbruksavvik

Hull telles **innenfor hver series observerte levetid**. Timer før første og etter
siste observasjon rapporteres separat mot hele filens tidsrom; de kan skyldes
kommuneendringer eller manglende kategorier. Det antas ikke at alle grupper finnes i alle kommuner.
Duplikate nøkler telles før tidsaksen gjøres unik. Slike rader utelates fra
beregningen av sprang og statistiske forbruksavvik for å unngå vilkårlig valg.

Forbruk per målepunkt vurderes innen samme kommune og forbruksgruppe.
En robust terskel (median + 10 × 1,4826 × MAD) gir kandidater til undersøkelse,
ikke bevis på feil. Den tar ikke eksplisitt høyde for temperatur eller sesong.
Private gjennomsnitt over 25 kWh per målepunkt per time får et eget varsel.
Industri har ingen felles fysisk øvre grense i denne kontrollen.


In [4]:
coverage_rows, gap_rows, jump_rows, outlier_rows, point_rows = [], [], [], [], []
key_duplicate_examples = []
global_first = min(a[0].min() for blocks in parts.values() for a in blocks)
global_last = max(a[0].max() for blocks in parts.values() for a in blocks)
global_hours = pd.date_range(pd.Timestamp(global_first, tz="UTC"), pd.Timestamp(global_last, tz="UTC"), freq="h")
global_seen = np.zeros(len(global_hours), dtype=bool)
daily_counts = defaultdict(Counter)

for (code, category), blocks in list(parts.items()):
    ts = pd.DataFrame({"t": np.concatenate([a[0] for a in blocks]),
                       "kwh": np.concatenate([a[1] for a in blocks]),
                       "points": np.concatenate([a[2] for a in blocks])}).sort_values("t")
    del parts[(code, category)]
    duplicated = ts["t"].duplicated(keep=False)
    excess = int(ts["t"].duplicated().sum())
    if excess:
        sample = ts.loc[duplicated].head(5).copy()
        sample["KOMMUNENUMMER"], sample["FORBRUKSGRUPPE"] = code, category
        key_duplicate_examples.append(sample)
    observed = pd.DatetimeIndex(pd.to_datetime(ts["t"].unique(), utc=True)).sort_values()
    on_hour = observed == observed.floor("h")
    aligned = observed[on_hour]
    global_seen |= global_hours.isin(aligned)
    expected = pd.date_range(observed.min().ceil("h"), observed.max().floor("h"), freq="h")
    absent = expected.difference(observed)
    coverage_rows.append((code, category, observed.min(), observed.max(), len(ts), len(observed), excess,
                          len(absent), int((global_hours < observed.min()).sum()), int((global_hours > observed.max()).sum())))
    if len(absent):
        # Én rad per sammenhengende hull, også for lange brudd.
        gap_groups = np.r_[True, np.diff(absent.asi8) != HOUR_NS].cumsum()
        for _, gap in pd.Series(absent).groupby(gap_groups):
            gap_rows.append((code, category, gap.iloc[0], gap.iloc[-1], len(gap)))
    days = pd.Series(aligned.tz_convert("Europe/Oslo").date).value_counts()
    for date, count in days.items():
        daily_counts[date][int(count)] += 1
    clean = ts.loc[~duplicated].copy()
    adjacent = clean["t"].diff().eq(HOUR_NS)
    previous = clean["points"].shift()
    delta = clean["points"] - previous
    jump = adjacent & previous.gt(0) & delta.abs().ge(POINT_CHANGE_MINIMUM) & delta.abs().div(previous).ge(POINT_CHANGE_FRACTION)
    for row in clean.loc[jump].index:
        jump_rows.append((code, category, pd.Timestamp(clean.loc[row, "t"], tz="UTC"), previous.loc[row], clean.loc[row, "points"], delta.loc[row]))
    valid_points = clean["points"].where(np.isfinite(clean["points"]) & clean["points"].ge(0) & clean["points"].mod(1).eq(0))
    point_rows.append((code, category, valid_points.min(), valid_points.median(), valid_points.max(), valid_points.isna().sum()))
    ratio = (clean["kwh"] / clean["points"].where(clean["points"].gt(0))).replace([np.inf, -np.inf], np.nan)
    median = ratio.median()
    mad = (ratio - median).abs().median()
    if pd.notna(mad) and mad > 0:
        high = ratio.gt(median + 10 * 1.4826 * mad)
        for row in clean.loc[high].index:
            outlier_rows.append((code, category, pd.Timestamp(clean.loc[row, "t"], tz="UTC"), clean.loc[row, "kwh"], clean.loc[row, "points"], ratio.loc[row], median + 10 * 1.4826 * mad))

coverage = pd.DataFrame(coverage_rows, columns=SERIES + ["første_utc", "siste_utc", "rader", "unike_timer", "duplikate_nøkler_ekstra", "manglende_timer_internt", "timer_før_start", "timer_etter_slutt"])
gaps = pd.DataFrame(gap_rows, columns=SERIES + ["fra_utc", "til_utc", "timer"])
point_summary = pd.DataFrame(point_rows, columns=SERIES + ["min", "median", "maks", "ugyldige_eller_manglende"])
point_jumps = pd.DataFrame(jump_rows, columns=SERIES + ["tid_utc", "før", "etter", "endring"])
consumption_outliers = pd.DataFrame(outlier_rows, columns=SERIES + ["tid_utc", "kwh", "målepunkter", "kwh_per_punkt", "terskel"])
global_missing_hours = global_hours[~global_seen]
print("Duplikate nøkler, overskytende rader:", coverage["duplikate_nøkler_ekstra"].sum())
print("Manglende timer internt, summert over serier:", len(gaps) and gaps["timer"].sum())
print("Timer helt uten observasjoner:", len(global_missing_hours))
print("Statistiske forbruksvarsler:", len(consumption_outliers))
print("Sprang i antall målepunkter:", len(point_jumps))
display(coverage.sort_values("manglende_timer_internt", ascending=False).head(20))
display(gaps.sort_values("timer", ascending=False).head(20))
display(point_summary.head(20))
display(point_jumps.head(20))
display(consumption_outliers.sort_values("kwh_per_punkt", ascending=False).head(20))
if key_duplicate_examples:
    display(pd.concat(key_duplicate_examples, ignore_index=True).head(20))


Duplikate nøkler, overskytende rader: 0
Manglende timer internt, summert over serier: 0
Timer helt uten observasjoner: 0
Statistiske forbruksvarsler: 86234
Sprang i antall målepunkter: 145


,KOMMUNENUMMER,FORBRUKSGRUPPE,første_utc,siste_utc,rader,unike_timer,duplikate_nøkler_ekstra,manglende_timer_internt,timer_før_start,timer_etter_slutt
0,1826,Forretning,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
1,1826,Industri,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
2,1826,Privat,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
3,1820,Forretning,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
4,1820,Industri,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
5,1820,Privat,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
6,5601,Forretning,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
7,5601,Industri,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
8,5601,Privat,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0
9,3428,Forretning,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,37199,37199,0,0,0,0


,KOMMUNENUMMER,FORBRUKSGRUPPE,fra_utc,til_utc,timer


,KOMMUNENUMMER,FORBRUKSGRUPPE,min,median,maks,ugyldige_eller_manglende
0,1826,Forretning,46.0,65.0,75.0,0
1,1826,Industri,19.0,34.0,41.0,0
2,1826,Privat,559.0,734.0,744.0,0
3,1820,Forretning,599.0,613.0,662.0,0
4,1820,Industri,134.0,155.0,182.0,0
5,1820,Privat,3862.0,3882.0,3908.0,0
6,5601,Forretning,1023.0,1145.0,1211.0,0
7,5601,Industri,410.0,420.0,447.0,0
8,5601,Privat,10252.0,10493.0,10782.0,0
9,3428,Forretning,185.0,191.0,208.0,0


,KOMMUNENUMMER,FORBRUKSGRUPPE,tid_utc,før,etter,endring
0,1826,Forretning,2022-10-16 22:00:00+00:00,48.0,65.0,17.0
1,1826,Industri,2022-10-16 22:00:00+00:00,23.0,38.0,15.0
2,1826,Privat,2022-10-16 22:00:00+00:00,559.0,720.0,161.0
3,3428,Industri,2022-10-16 22:00:00+00:00,46.0,65.0,19.0
4,4631,Forretning,2022-12-18 23:00:00+00:00,1279.0,1973.0,694.0
5,1871,Industri,2022-11-30 23:00:00+00:00,111.0,140.0,29.0
6,4203,Industri,2022-12-31 23:00:00+00:00,860.0,685.0,-175.0
7,4632,Forretning,2022-12-18 23:00:00+00:00,168.0,264.0,96.0
8,5520,Industri,2022-10-16 22:00:00+00:00,79.0,105.0,26.0
9,1839,Forretning,2022-12-31 23:00:00+00:00,102.0,134.0,32.0


,KOMMUNENUMMER,FORBRUKSGRUPPE,tid_utc,kwh,målepunkter,kwh_per_punkt,terskel
53526,5059,Industri,2026-04-14 18:00:00+00:00,1.256062e+07,617.0,20357.567287,236.003277
53522,5059,Industri,2026-04-14 14:00:00+00:00,1.256051e+07,617.0,20357.398575,236.003277
53523,5059,Industri,2026-04-14 15:00:00+00:00,1.256029e+07,617.0,20357.031937,236.003277
53524,5059,Industri,2026-04-14 16:00:00+00:00,1.256028e+07,617.0,20357.017574,236.003277
53525,5059,Industri,2026-04-14 17:00:00+00:00,1.256007e+07,617.0,20356.680486,236.003277
53527,5059,Industri,2026-04-14 19:00:00+00:00,1.255976e+07,617.0,20356.170355,236.003277
53529,5059,Industri,2026-04-14 21:00:00+00:00,1.255869e+07,617.0,20354.443776,236.003277
53528,5059,Industri,2026-04-14 20:00:00+00:00,1.255639e+07,617.0,20350.705313,236.003277
86224,4643,Industri,2022-08-31 20:00:00+00:00,4.019446e+05,58.0,6930.080052,6714.637070
86218,4643,Industri,2022-08-31 14:00:00+00:00,3.990492e+05,58.0,6880.157862,6714.637070


## Sommertid

Norske lokale døgn ved tidsskiftet har 23 eller 25 timer. På høsten er to lokale
klokkeslett 02:00 forskjellige observasjoner når UTC-offseten er forskjellig.
En sammenhengende UTC-serie skal derfor fortsatt ha én observasjon hver time.
Tabellen viser antall serier med komplett dekning på overgangsdøgnet.
Første/siste ufullstendige døgn og serier med kortere levetid kan gi avvik.

Kilde: [Forskrift om sommertid, forklart av Lovdata](https://lovdata.no/artikkel/natt_til_sondag_stilles_klokkene_/5454).


In [5]:
first_local = global_hours[0].tz_convert("Europe/Oslo").normalize()
last_local = global_hours[-1].tz_convert("Europe/Oslo").normalize()
midnights = pd.date_range(first_local, last_local + pd.DateOffset(days=1), freq="D")
dst_rows = []
for start, end in zip(midnights[:-1], midnights[1:]):
    hours = int((end.tz_convert("UTC") - start.tz_convert("UTC")) / pd.Timedelta(hours=1))
    if hours != 24:
        counts = daily_counts.get(start.date(), Counter())
        dst_rows.append((start.date(), hours, counts.get(hours, 0), sum(counts.values()) - counts.get(hours, 0), dict(sorted(counts.items()))))
dst_report = pd.DataFrame(dst_rows, columns=["lokal_dato", "forventede_timer", "komplette_serier", "ufullstendige_serier", "timer_per_observert_serie"])
display(dst_report)
print("Feil UTC-offset mot Europe/Oslo:", {k: v for k, v in issues.items() if "offset" in k})


Feil UTC-offset mot Europe/Oslo: {'feil_oslo_offset_STARTTID': 0, 'feil_oslo_offset_SLUTTID': 0}


,lokal_dato,forventede_timer,komplette_serier,ufullstendige_serier,timer_per_observert_serie
0,2022-03-27,23,1074,0,{23: 1074}
1,2022-10-30,25,1074,0,{25: 1074}
2,2023-03-26,23,1074,0,{23: 1074}
3,2023-10-29,25,1074,0,{25: 1074}
4,2024-03-31,23,1074,0,{23: 1074}
5,2024-10-27,25,1074,0,{25: 1074}
6,2025-03-30,23,1074,0,{23: 1074}
7,2025-10-26,25,1074,0,{25: 1074}
8,2026-03-29,23,1074,0,{23: 1074}


## Kommunenummer, navn og kommunegrenser

SSB beskriver nye kommunenummer fra 1. januar 2024 og delingen av **1507 Ålesund**
i **1508 Ålesund** og **1580 Haram**. Et nytt nummer trenger ikke bety nye grenser.
Omvendt kan mindre grensejusteringer forekomme uten nytt nummer.

Tabellene viser hvilke koder og navn som faktisk finnes i filen, levetiden deres,
navn med flere koder og koder med flere navn. Vi undersøker også om Ålesund/Haram
følger historiske koder eller om nyere koder er brukt bakover i tid.
CSV-en alene kan **ikke** bekrefte geografien eller om leverandøren har harmonisert
historikken. Dette må avklares med leverandørens metadata og SSBs korrespondanser
for de aktuelle årene før tidsserier skjøtes. Ingen automatisk omkoding eller
fordeling av tidligere Ålesund-forbruk på nye kommuner gjøres her.

Kilder: [SSB: Historikk ved ny kommune- og fylkesstruktur](https://www.ssb.no/statbank/hvordan-bruke-statistikkbanken/hvordan-ivareta-historikken),
[SSB: Alle endringer i regionale inndelinger](https://www.ssb.no/metadata/alle-endringer-i-de-regionale-inndelingene),
[Regjeringen: Nye kommune- og fylkesnummer fra 2024](https://www.regjeringen.no/no/tema/kommuner-og-regioner/kommunestruktur/nye-kommune-og-fylkesnummer-fra-1.-januar-2024/id2924701/).


In [6]:
municipalities = pd.concat(geography_parts, ignore_index=True).groupby(
    ["KOMMUNENUMMER", "KOMMUNE"], dropna=False
).agg(første_utc=("min", "min"), siste_utc=("max", "max"), rader=("size", "sum")).reset_index()
name_codes = municipalities.groupby("KOMMUNE")["KOMMUNENUMMER"].nunique()
code_names = municipalities.groupby("KOMMUNENUMMER")["KOMMUNE"].nunique()
print("Unike kommunenummer:", municipalities["KOMMUNENUMMER"].nunique())
print("Unike kommunenavn:", municipalities["KOMMUNE"].nunique())
display(municipalities)
display(municipalities[municipalities["KOMMUNE"].isin(name_codes.index[name_codes.gt(1)])])
display(municipalities[municipalities["KOMMUNENUMMER"].isin(code_names.index[code_names.gt(1)])])
alesund_haram = municipalities[municipalities["KOMMUNENUMMER"].isin(["1507", "1508", "1580"])].copy()
reform = pd.Timestamp("2024-01-01", tz="Europe/Oslo").tz_convert("UTC")
alesund_haram["ny_kode_observert_før_2024"] = alesund_haram["KOMMUNENUMMER"].isin(["1508", "1580"]) & alesund_haram["første_utc"].lt(reform)
alesund_haram["gammel_kode_observert_fra_2024"] = alesund_haram["KOMMUNENUMMER"].eq("1507") & alesund_haram["siste_utc"].ge(reform)
display(alesund_haram)
# Koder som starter/slutter nær årsskifter er kandidater til videre kontroll.
display(coverage.loc[
    coverage["første_utc"].dt.tz_convert("Europe/Oslo").dt.month.eq(1)
    | coverage["siste_utc"].dt.tz_convert("Europe/Oslo").dt.month.eq(12),
    SERIES + ["første_utc", "siste_utc", "timer_før_start", "timer_etter_slutt"]
].drop_duplicates())


Unike kommunenummer: 358
Unike kommunenavn: 356


,KOMMUNENUMMER,KOMMUNE,første_utc,siste_utc,rader
0,0000,Ukjent tilhørighet,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
1,0301,Oslo,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
2,1101,Eigersund,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
3,1103,Stavanger,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
4,1106,Haugesund,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
...,...,...,...,...,...
353,5628,Deatnu,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
354,5630,Berlevåg,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
355,5632,Båtsfjord,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
356,5634,Vardø,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597


,KOMMUNENUMMER,KOMMUNE,første_utc,siste_utc,rader
30,1515,Herøy,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
59,1818,Herøy,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
99,3114,Våler,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597
156,3419,Våler,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597


,KOMMUNENUMMER,KOMMUNE,første_utc,siste_utc,rader


,KOMMUNENUMMER,KOMMUNE,første_utc,siste_utc,rader,ny_kode_observert_før_2024,gammel_kode_observert_fra_2024
27,1508,Ålesund,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597,True,False
51,1580,Haram,2022-01-31 23:00:00+00:00,2026-04-30 21:00:00+00:00,111597,True,False


,KOMMUNENUMMER,FORBRUKSGRUPPE,første_utc,siste_utc,timer_før_start,timer_etter_slutt


## Oppsummering og videre behandling

Ingen rader er fjernet eller fylt inn. Undersøk eksemplene før korrigering.
Manglende timer må ikke automatisk settes til null; et hull er ikke et observert nullforbruk.
Sprang i målepunkter kan skyldes dekning, kundemasse eller kommuneendringer.
Rapportene `coverage`, `gaps`, `point_summary`, `point_jumps`,
`consumption_outliers`, `dst_report` og `municipalities` kan brukes videre i analysen.
